# 03 – pandas-Basics

pandas verarbeitet tabellarische Daten. Die wichtigste Struktur ist der `DataFrame`: Zeilen entsprechen häufig Beobachtungen, Spalten den Merkmalen.

## Lernziele

- `Series` und `DataFrame` unterscheiden,
- Daten auswählen, filtern und sortieren,
- fehlende Werte behandeln,
- Daten gruppieren und neue Merkmale erzeugen,
- Dateien einlesen und speichern.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)
print("pandas-Version:", pd.__version__)

## 1. Series und DataFrame

In [ ]:
punkte = pd.Series([72, 88, 91], index=["Ada", "Ben", "Cem"], name="Punkte")
display(punkte)

daten = pd.DataFrame({
    "name": ["Ada", "Ben", "Cem", "Dana", "Eli", "Fatma"],
    "gruppe": ["A", "B", "A", "B", "A", "B"],
    "alter": [29, 34, 41, 25, np.nan, 38],
    "lernstunden": [6.5, 3.0, 8.0, 4.5, 7.0, 5.5],
    "punkte": [82, 65, 94, 71, 88, 79],
})
display(daten)

## 2. Einen Datensatz kennenlernen

In [ ]:
print("Form:", daten.shape)
print("Spalten:", daten.columns.tolist())
print("Datentypen:\n", daten.dtypes)
display(daten.head(3))
display(daten.select_dtypes(include="number").describe())

`head()` zeigt erste Zeilen, `shape` liefert `(Zeilen, Spalten)`, `dtypes` die Datentypen und `describe()` deskriptive Kennzahlen. Bei einem unbekannten Datensatz gehören diese Prüfungen an den Anfang.

## 3. Spalten und Zeilen auswählen

In [ ]:
display(daten["punkte"])                         # eine Spalte -> Series
display(daten[["name", "punkte"]])               # mehrere Spalten -> DataFrame
display(daten.loc[1:3, ["name", "gruppe"]])       # labelbasiert, Ende inklusive
display(daten.iloc[1:4, 0:2])                    # positionsbasiert, Ende exklusiv

## 4. Filtern und sortieren

In [ ]:
fleißig_und_erfolgreich = daten[(daten["lernstunden"] >= 6) & (daten["punkte"] >= 80)]
display(fleißig_und_erfolgreich)

display(daten.sort_values("punkte", ascending=False))

Bei kombinierten Bedingungen stehen die Einzelbedingungen in Klammern. pandas verwendet `&` für UND, `|` für ODER und `~` für NICHT.

## 5. Fehlende Werte

In [ ]:
print(daten.isna().sum())

daten_bereinigt = daten.copy()
median_alter = daten_bereinigt["alter"].median()
daten_bereinigt["alter"] = daten_bereinigt["alter"].fillna(median_alter)
display(daten_bereinigt)

Fehlende Werte werden nicht automatisch immer gelöscht. Mögliche Strategien sind Entfernen, Ersetzen (Imputation) oder ein eigenes Merkmal für das Fehlen. Die Wahl hängt vom Datensatz und der späteren Nutzung ab. `copy()` verhindert hier, dass der ursprüngliche DataFrame verändert wird.

## 6. Neue Spalten und Gruppierungen

In [ ]:
daten_bereinigt["bestanden"] = daten_bereinigt["punkte"] >= 70
daten_bereinigt["punkte_pro_stunde"] = (
    daten_bereinigt["punkte"] / daten_bereinigt["lernstunden"]
)
display(daten_bereinigt)

gruppen_statistik = (
    daten_bereinigt
    .groupby("gruppe", as_index=False)
    .agg(
        anzahl=("name", "count"),
        mittlere_punkte=("punkte", "mean"),
        mittlere_lernstunden=("lernstunden", "mean"),
    )
)
display(gruppen_statistik)

## 7. Ein einfaches Diagramm

In [ ]:
import matplotlib.pyplot as plt

ax = daten_bereinigt.plot.scatter(
    x="lernstunden", y="punkte", s=70, title="Lernstunden und Punkte"
)
ax.set_xlabel("Lernstunden")
ax.set_ylabel("Punkte")
ax.grid(alpha=0.3)
plt.show()

Ein Diagramm kann Zusammenhänge sichtbar machen, beweist aber keine Kausalität: Mehr Lernstunden können mit höheren Punkten zusammenhängen, ohne allein deren Ursache zu sein.

## 8. Dateien einlesen und speichern

Die häufigsten Befehle sind:

```python
df = pd.read_csv("daten.csv")
df.to_csv("ergebnis.csv", index=False)
```

`index=False` verhindert eine zusätzliche Indexspalte in der Ausgabedatei. Bei deutschen CSV-Dateien können `sep=";"` und `decimal=","` nötig sein. Relative Pfade beziehen sich auf den Arbeitsordner des Kernels.

## Übung

1. Filtere alle Personen aus Gruppe B.
2. Berechne deren durchschnittliche Punktzahl.
3. Sortiere das Ergebnis nach den Lernstunden absteigend.

In [ ]:
# Musterlösung
gruppe_b = daten_bereinigt.loc[daten_bereinigt["gruppe"] == "B"].copy()
print("Durchschnittliche Punkte:", gruppe_b["punkte"].mean())
display(gruppe_b.sort_values("lernstunden", ascending=False))

## Merksätze

- `Series` ist eindimensional, `DataFrame` zweidimensional.
- Erst Daten prüfen, dann bereinigen und analysieren.
- `.loc` arbeitet mit Labels, `.iloc` mit Positionen.
- Fehlende Werte brauchen eine begründete Strategie.
- Transformationen möglichst nachvollziehbar und reproduzierbar halten.